# R1 · Ingesta y arquitectura

**Responsable:** Nicolás (Rol A)  
**Revisor:** Cristian (Rol C)

## Qué queda resuelto

- [x] Descargar los 1.143 archivos diarios, series, UID, OWID y Banco Mundial (`src/01_ingesta_y_unificacion.py`)
- [x] Unificar los 5 esquemas y mostrar filas por esquema
- [x] Guardar en Parquet y comparar con CSV (tamaño y tiempo de lectura)
- [x] Justificar motor relacional/analítico (DuckDB / PostgreSQL) frente a NoSQL (S4)

In [ ]:
import sys, pathlib, glob, os, time
RAIZ = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(RAIZ))
import pandas as pd

RAW_DAILY = RAIZ / 'data' / 'raw' / 'daily_reports'
PARQUET_PATH = RAIZ / 'data' / 'clean' / 'reportes_unificados.parquet'

## Verificar los 1.143 archivos diarios descargados

In [ ]:
csvs = sorted(RAW_DAILY.glob('*.csv'))
print('Archivos diarios en data/raw/daily_reports:', len(csvs))
print('Primero:', csvs[0].name, '| Último:', csvs[-1].name)

## Unificación de los 5 esquemas y filas por esquema

In [ ]:
t0 = time.perf_counter()
df_pq = pd.read_parquet(PARQUET_PATH)
t_pq = time.perf_counter() - t0
print('Total de filas unificadas:', len(df_pq))
display(df_pq.groupby('esquema', as_index=False).agg(filas=('fecha_reporte', 'count'), desde=('fecha_reporte', 'min'), hasta=('fecha_reporte', 'max')))

## Comparación CSV vs. Parquet (tamaño y tiempo de lectura)

In [ ]:
tam_csv_mb = sum(p.stat().st_size for p in csvs) / (1024 ** 2)
tam_pq_mb = PARQUET_PATH.stat().st_size / (1024 ** 2)
t0 = time.perf_counter()
for p in csvs[:100]:
    pd.read_csv(p, low_memory=False)
t_csv_est = (time.perf_counter() - t0) * (len(csvs) / 100.0)
display(pd.DataFrame([
    {'formato': 'CSV (1.143 archivos)', 'tamano_mb': round(tam_csv_mb, 2), 'lectura_s': round(t_csv_est, 2)},
    {'formato': 'Parquet (capa limpia)', 'tamano_mb': round(tam_pq_mb, 2), 'lectura_s': round(t_pq, 2)},
]))

## Justificación del motor elegido (DuckDB / PostgreSQL) frente a NoSQL (S4)

Una vez unificados los 5 esquemas en `snake_case`, los 4,28 millones de registros tienen esquema tabular estable, integridad referencial hacia dimensiones (`dim_fecha`, `dim_territorio`) y carga analítica OLAP (`JOIN`, funciones ventana `LAG` para calcular casos nuevos y agregaciones `GROUP BY` / `ROLLUP`). Motores relacionales columnares/analíticos como DuckDB (o PostgreSQL para concurrencia multiusuario) están optimizados para estas consultas, mientras bases NoSQL documentales (MongoDB) o de escritura masiva (Cassandra) no aportan ventajas y penalizan los cruces analíticos (ver `docs/arquitectura.md`).